# Script for processing supplemental data : 

In [1]:
import os
import pandas as pd
import neurokit2 as nk
import matplotlib.pyplot as plt
import json
import numpy as np

In [2]:
input_folder: str = r"C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Biosignaler til L&K"
output_base  = r"C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data"

In [3]:
# define categories 
# -> based on experimental conditions 
categories = ["baseline" , "VR" , "post"]
for cat in categories :
    
    # create one folder for each category, inside the output directory 
    # exist_ok=True to prevent erros if the folder already exists 
    os.makedirs(os.path.join(output_base , cat) , exist_ok = True)

In [4]:
# find .txt files in the input folder 
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

In [5]:
# Categorising the files
# loop through each of the files, determine category from the filename 
for file in txt_files :
    # combine filename with input folder to get path for the file 
    file_path = os.path.join(input_folder , file)
    
    # read everything in lowercase to prevent errors 
    fname_lower = file.lower()
    
    # too many files, takes forever to run, if we don't already pick out the converted files here 
    if "converted"  not in fname_lower: 
        continue 

    # sort into folder based on filename 
    if "baseline" in fname_lower :
        category = "baseline"
    elif "stim" in fname_lower : 
        category = "VR"
    elif "post" in fname_lower or "pr" in fname_lower :
        category = "post" 
    else :
        # if none of the defines categories exist in filename, skip the file 
        print(f"Skipping {file} : no category detected.")
        continue

    # Preparing the files for processing 
    # try/except for error handling 
    try :
        # read file, ignore lines starting with # (metadata lines)
        df = pd.read_csv(
            file_path ,
            sep = "\t" ,        # split into horizontal strings 
            comment = "#" ,     # skip lines starting with # (metadata)
            header = None ,     # we remove old header entirely # TODO: slå lige op om "comment" og "header" laver dobbeltkonfekt 
            names = ["trash1" , "trash2" , "trash3" , "ECG" , "trash4" , "trash5" , "trash6"] ,   # manually rename columns  
            engine = "python"   # TODO : er den nødvendig ? 
        )

        with open(file_path, "r") as infile :
            # use json to store necessary metadata from the header 
            # -> we need the sampling rate for future calculations 
            info = json.loads(infile.readlines()[1].replace("# ", "").replace("\n", ""))
    
        # Extracting metadata 
        # keep the ECG column
        df = df[["ECG"]] 

        # save the sampling rate 
        # find this in the data saved from the header 
        sampling_rate = info[list(info.keys())[0]]["sampling rate"]

        # Data processing 
        # convert to numeric (from string)
        # coerce to deal if something cannot be converted (replace w NaN)
        df["ECG"] = pd.to_numeric(df["ECG"] , errors = "coerce") 

        # a time vector is established (in secs)
        # sampling rate is used to calculate time vector 
        df["Time"] = np.arange(0, len(df["ECG"])) / sampling_rate  
        

        # Save processed files 
        # Output path
        output_csv = os.path.join(
            output_base ,
            category ,
            # change .txt files into .csv files
            file.replace(".txt" , ".csv")
        )

        # save CSV, without index column 
        df.to_csv(output_csv , index = False) 

        print(f"Saved {output_csv}")

    except Exception as e :
        print(f"Error processing {file} : {e}")

Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\post\VR20_PR2_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\baseline\vrP10_baseline1_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\baseline\vrP10_baseline2_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\post\vrP10_PR1_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\post\vrP10_PR2_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\VR\vrP10_stim1_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\VR\vrP10_stim2_converted.csv
Saved C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed

## Part 2 : process the files, after they have been sorted 

In [6]:
# define a function for calculations 
def calcs(results , signal , sampling_rate , state , segment , file) -> None :

    # process signal: clean signal, find R-peaks (eg_peaks()), find HR (using signal_rate())
    # signals : DataFrame w raw signal, clean signal, heart rates 
    # info: dict with R-peak locations and sampling rate 
    signals, info = nk.ecg_process(signal, sampling_rate = sampling_rate)

    # calculate HR mean from the processed signal 
    heart_rate_mean = float(signals["ECG_Rate"].mean())

    # calculate time-domain variables for HRV 
    # show=False, we don't need plots 
    hrv = nk.hrv_time(info , sampling_rate = sampling_rate , show = False)

    results.append({
        "State" : state ,
        "Participant" : file.split(".")[0] ,
        "Segment" : segment ,

        # Only [0], not [0][0,0] , no more nested arrays, just columns 
        "HeartRate_Mean" : heart_rate_mean,
       "HRV_Sdnn" : float(hrv["HRV_SDNN"][0]) ,
        "HRV_rmssd" : float(hrv["HRV_RMSSD"][0]) ,
        "Hrv_pnn50" : float(hrv["HRV_pNN50"][0]) ,
        "Hrv_meanNN" : float(hrv["HRV_MeanNN"][0]) ,
        "Hrv_medianNN" : float(hrv["HRV_MedianNN"][0]) ,
    })


In [ ]:
# retning / mappe med CSV filerne 
# -> should have a folder for each of the three cats  
base_path = r"C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data"

## folder names / hvilken type måling 
states = ["baseline", "VR","post"]

sec_pr_min = 60 # no magic numbers

# zero overlap, too many files to justify extra runtime 
overlap = 0
 
# loop through each folder  
for state in states :
 
    # create a list to hold the results for the current folder
    # -> will reset for each folder / state 
    results = []

    # combine path w file name for file path 
    state_path = os.path.join(base_path, state)

    # set up conditions for the files we want to analyse 
    # has to be .csv file 
    # must have converted in the name (to avoid doubles)
    # cannot have analysis in the name 
    files = [f for f in os.listdir(state_path)
             if f.endswith(".csv") and "converted" in f and "analysis" not in f]
 
    # loop through each file in the folder 
    for file in files:
        file_path = os.path.join(state_path, file)
        
        # read the CSV file into a DataFrame 
        df = pd.read_csv(file_path)

        # if the file has no ECG comlumn, skip it, print message 
        if "ECG" not in df.columns :
            print(f"Skipping {file} as it does not contain 'ECG' column.")
            continue
 
        # extract ECG signal 
        # dropna() : remove empty vals 
        # reset_index(drop=True) : giver signal index from 0 
        ecg_signal = df["ECG"].dropna().reset_index(drop = True) 


        # keep only the first 5 minutes of the signal
        max_duration_sec = 5 * 60
        max_samples = int(sampling_rate * max_duration_sec)
        ecg_signal = ecg_signal.iloc[:max_samples].reset_index(drop = True)


        try :
            # first we try running the calcs for the full signal 
            calcs(results , ecg_signal , sampling_rate , state , "Full" , file)
        except Exception as e :
            print(f"Error processing full signal for {file} : {e}")
            continue

        # we split into intervals 
        if True : 
            
            # number of samples , make sure its an int
            # shape: antal elementer i hver dim 
            Nsamples: int = ecg_signal.shape[0] 

            # number of samples pr interval 
            interval_length = int(sampling_rate * sec_pr_min)

            # "step size" between each interval 
            # take overlap into account 
            # interval allows for smoother cals
            # don't risk losing data between the two split points 
            interval_changes = int(interval_length * (1 - overlap))

            Nmin: int = int(Nsamples // interval_length) # Number of minutes 

            N_intervals = int(((Nsamples - interval_length) // interval_changes) + 1)

            # analyse each interval 
            for i in range(N_intervals) :

                # start + end sample for current interval 
                # takes overlap into account 
                start_idx = i * interval_changes
                end_idx = start_idx + interval_length

                # if the interval exceeds the length of the interval, skip/move on 
                if end_idx > len(ecg_signal) :
                    print(f"Skipping interval {i+1} for {file}")
                    continue

                # extract ECG for the interval
                minute_signal = ecg_signal.iloc[start_idx:end_idx].reset_index(drop = True)  

                # now try running the calculations for the intervals 
                # also label and add to output file 
                try :
                    calcs(results, minute_signal, sampling_rate, state, f"Interval_{i+1}", file)
                except Exception as e :
                    print(f"Error processing interval {i+1} for {file} : {e}") 
 
    # when all files in folder/state have been processed, convert collected data into DataFrame
    results_df = pd.DataFrame(results) 
    output_path = os.path.join(state_path , "ecg_analysis_results.csv")

    # save without row index
    results_df.to_csv(
        output_path ,
        index = False ,
        float_format="%.0f" 
    )
 
    print(f"Results saved to {output_path}")


Results saved to C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\baseline\ecg_analysis_results.csv
Results saved to C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\VR\ecg_analysis_results.csv
Results saved to C:\Users\Laerke\Documents\DTU\Teknisk Biomedicin\SS 26\BSc\EKG\Processed signals, Ashers Data\post\ecg_analysis_results.csv
